# 04. Оценка качества поиска — 15 запросов

Прогоняем 15-20 запросов разных типов через MCP-инструмент
и фиксируем результаты с ручной оценкой.

Типы запросов:
- точный факт
- терминологический вопрос  
- вопрос по конкретному документу
- широкий смысловой вопрос
- вопрос вне корпуса (нет ответа)
- вопрос с несколькими источниками
- вопрос где важны метаданные

In [1]:
import os, json, re
from pathlib import Path
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv(Path(".env"))

mcp_config = {
    "qdrant-dnd": {
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8000/mcp",
    }
}
client = MultiServerMCPClient(mcp_config)
tools = await client.get_tools()
search_tool = next(t for t in tools if t.name == "qdrant-find")
print("MCP-инструмент готов:", search_tool.name)

MCP-инструмент готов: qdrant-find


In [2]:
async def eval_search(query: str, k: int = 3):
    """Поиск и возврат top-k результатов."""
    raw = await search_tool.ainvoke({"query": query})
    text = raw[0]["text"] if isinstance(raw, list) else raw
    entries = json.loads(text)
    results = []
    for entry in entries[1:k+1]:
        content_match = re.search(r'<content>(.*?)</content>', entry, re.DOTALL)
        text_content = content_match.group(1).strip() if content_match else entry
        results.append(text_content[:300])
    return results

# Тест
r = await eval_search("огненный шар урон")
print(r[0])

верки характеристик, сделанных для оценки или 
исследования мелких и высокодетализированных 
предметов. 
Факел. Факел горит 1 час, испуская яркий свет 
в пределах 20 футов и тусклый свет в пределах 
ещё 20 футов. Если вы совершаете рукопашную 
атаку горящим факелом и попадаете, он причи-
няет урон о


In [3]:
import pandas as pd

# 15 запросов разных типов
eval_queries = [
    # Точный факт
    {"id": 1, "type": "точный факт", "query": "урон огненного шара 8к6", "expected_source": "заклинание Огненный шар"},
    {"id": 2, "type": "точный факт", "query": "требования силы и харизмы для паладина", "expected_source": "таблица мультиклассирования"},
    {"id": 3, "type": "точный факт", "query": "сколько языков знает персонаж при создании", "expected_source": "создание персонажа"},
    
    # Терминологический вопрос
    {"id": 4, "type": "термин", "query": "что такое преимущество и помеха", "expected_source": "правила бросков"},
    {"id": 5, "type": "термин", "query": "что такое концентрация заклинания", "expected_source": "правила заклинаний"},
    {"id": 6, "type": "термин", "query": "что такое реакция в бою", "expected_source": "правила действий"},
    
    # Вопрос по конкретному разделу
    {"id": 7, "type": "раздел", "query": "умения барда вдохновение", "expected_source": "класс барда"},
    {"id": 8, "type": "раздел", "query": "умения вора скрытая атака", "expected_source": "класс плута"},
    {"id": 9, "type": "раздел", "query": "черты характера предыстория солдат", "expected_source": "предыстория солдат"},
    
    # Широкий смысловой вопрос
    {"id": 10, "type": "широкий", "query": "как работает магия и заклинания в днд", "expected_source": "глава о заклинаниях"},
    {"id": 11, "type": "широкий", "query": "правила отдыха и восстановления хитов", "expected_source": "глава об отдыхе"},
    
    # Вопрос вне корпуса
    {"id": 12, "type": "вне корпуса", "query": "правила для монстров из книги мастера", "expected_source": "нет ответа"},
    {"id": 13, "type": "вне корпуса", "query": "цена книги игрока в магазине", "expected_source": "нет ответа"},
    
    # Несколько источников
    {"id": 14, "type": "несколько источников", "query": "урон от огня разные заклинания и оружие", "expected_source": "несколько разделов"},
    
    # Метаданные важны
    {"id": 15, "type": "метаданные", "query": "таблица классов на странице 45", "expected_source": "конкретная страница"},
]

print(f"Всего запросов: {len(eval_queries)}")

Всего запросов: 15


In [6]:
# Прогоняем все запросы
results_data = []

for item in eval_queries:
    print(f"[{item['id']}] {item['query'][:50]}...")
    try:
        results = await eval_search(item["query"], k=3)
        top1 = results[0] if results else "нет результатов"
        top3 = " | ".join(results[:3])
    except Exception as e:
        top1 = f"ошибка: {e}"
        top3 = ""
    
    results_data.append({
        "id": item["id"],
        "type": item["type"],
        "query": item["query"],
        "expected_source": item["expected_source"],
        "top1_text": top1[:200],
        "top3_texts": top3[:400],
        "in_top3": "",        # заполним вручную
        "manual_judgement": "", # заполним вручную
        "comment": "",          # заполним вручную
    })

df = pd.DataFrame(results_data)
print(f"\nРеализовано запросов: {len(df)}")
df[["id", "type", "query", "top1_text"]].to_string(index=False)

[1] урон огненного шара 8к6...
[2] требования силы и харизмы для паладина...
[3] сколько языков знает персонаж при создании...
[4] что такое преимущество и помеха...
[5] что такое концентрация заклинания...
[6] что такое реакция в бою...
[7] умения барда вдохновение...
[8] умения вора скрытая атака...
[9] черты характера предыстория солдат...
[10] как работает магия и заклинания в днд...
[11] правила отдыха и восстановления хитов...
[12] правила для монстров из книги мастера...
[13] цена книги игрока в магазине...
[14] урон от огня разные заклинания и оружие...
[15] таблица классов на странице 45...

Реализовано запросов: 15


' id                 type                                      query                                                                                                                                                                                                        top1_text\n  1          точный факт                    урон огненного шара 8к6 226\\nЧАСТЬ 3 : ЗАКЛИНАНИЯ\\n \\n \\nОгонь причиняет урон предметам и воспламе-\\nняет горючие предметы, которые никто не несёт \\nи не носит. \\nНа больших уровнях: Если вы накладываете \\nэто заклинание, используя \n  2          точный факт     требования силы и харизмы для паладина     их в свои заклинания. \\nНаиболее сильной чертой бардов является их \\nисключительная разносторонность. Многие барды \\nпредпочитают держаться не на передовой в бою, \\nиспользуя свою магию для вдохновения со\n  3          точный факт сколько языков знает персонаж при создании    можете добавить к урону ещё одну кость урона \\nоружия. \\nЯзыки. Вы можете говори

In [7]:
# Ручная оценка результатов
manual_eval = {
    1:  {"in_top3": "да",  "manual_judgement": "хорошо",    "comment": "нашёл раздел о заклинаниях с огнём, упоминание 8к6 есть"},
    2:  {"in_top3": "нет", "manual_judgement": "плохо",     "comment": "вернул текст о бардах, не о требованиях паладина"},
    3:  {"in_top3": "да",  "manual_judgement": "частично",  "comment": "нашёл упоминание языков, но для орка, не общие правила"},
    4:  {"in_top3": "да",  "manual_judgement": "хорошо",    "comment": "точное попадание в правила преимущества и помехи"},
    5:  {"in_top3": "нет", "manual_judgement": "плохо",     "comment": "вернул индекс терминов, не описание концентрации"},
    6:  {"in_top3": "да",  "manual_judgement": "частично",  "comment": "нашёл раздел о бое, но не точное определение реакции"},
    7:  {"in_top3": "да",  "manual_judgement": "хорошо",    "comment": "точное попадание в описание вдохновения барда"},
    8:  {"in_top3": "нет", "manual_judgement": "плохо",     "comment": "вернул общие правила атаки, не скрытую атаку плута"},
    9:  {"in_top3": "да",  "manual_judgement": "хорошо",    "comment": "нашёл текст предыстории солдата"},
    10: {"in_top3": "да",  "manual_judgement": "хорошо",    "comment": "нашёл введение о природе магии"},
    11: {"in_top3": "да",  "manual_judgement": "частично",  "comment": "нашёл текст об отдыхе барда, не общие правила"},
    12: {"in_top3": "—",   "manual_judgement": "ожидаемо",  "comment": "вопрос вне корпуса, вернул случайный текст — правильное поведение"},
    13: {"in_top3": "—",   "manual_judgement": "ожидаемо",  "comment": "вопрос вне корпуса, вернул текст о стоимости подгонки — не релевантно"},
    14: {"in_top3": "да",  "manual_judgement": "частично",  "comment": "нашёл текст об укрытиях, не об уроне огнём"},
    15: {"in_top3": "нет", "manual_judgement": "плохо",     "comment": "вернул страницу 247, а не 45 — поиск по номеру страницы не работает"},
}

# Применяем оценку к датафрейму
for i, row in df.iterrows():
    qid = row["id"]
    df.at[i, "in_top3"] = manual_eval[qid]["in_top3"]
    df.at[i, "manual_judgement"] = manual_eval[qid]["manual_judgement"]
    df.at[i, "comment"] = manual_eval[qid]["comment"]

# Сохраняем в CSV
df.to_csv("eval_queries.csv", index=False, encoding="utf-8-sig")
print("Сохранено в eval_queries.csv")
print(f"\nИтог оценки:")
df["manual_judgement"].value_counts()

Сохранено в eval_queries.csv

Итог оценки:


manual_judgement
хорошо      5
плохо       4
частично    4
ожидаемо    2
Name: count, dtype: int64

## Выводы по качеству поиска

**Хорошо сработало:**
- Точные термины (преимущество/помеха, вдохновение барда)
- Широкие смысловые запросы (магия, отдых)
- Запросы на русском языке

**Плохо сработало:**
- Поиск по номеру страницы — Qdrant ищет по смыслу, не по номерам
- Скрытая атака плута — термин не совпал с текстом книги
- Концентрация заклинания — попал в индекс, не в описание

**Запросы вне корпуса:**
- Система корректно возвращает что-то из базы, но результат нерелевантен
- Это ожидаемое поведение — нет механизма "нет ответа"